In [1]:

from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("VideoGameStreaming")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN") # solo mostrar mensajes de avisos

In [2]:
game_schema = StructType([
    StructField("event_id", IntegerType()),
    StructField("timestamp", TimestampType()),
    StructField("user_id", StringType()),
    StructField("event_type", StringType()),
    StructField("game_mode", StringType()),
    StructField("level", IntegerType()),
    StructField("score", IntegerType()),
    StructField("duration_seconds", IntegerType()),
    StructField("country", StringType())
])

In [4]:
games_stream = (
    spark.readStream
    .option("header", True)
    .schema(game_schema)
    .csv("/opt/spark-data/streaming/incoming")
)

In [5]:
print(games_stream.isStreaming)

True


In [6]:
match_starts = games_stream.filter(
    col("event_type") == "match_start"
)

In [7]:
query = (
    match_starts.writeStream
    .format("parquet")
    .outputMode("append")
    .option(
        "path",
        "/opt/spark-data/streaming/output"
    )
    .option(
        "checkpointLocation",
        "/opt/spark-data/streaming/checkpoint"
    )
    .trigger(processingTime="5 seconds")
    .start()
)

In [9]:
print(query.isActive)
print(query.status)
print(query.lastProgress)

True
{'message': 'Processing new data', 'isDataAvailable': True, 'isTriggerActive': True}
None


In [44]:
query.stop()

In [12]:
results = spark.read.parquet(
    "/opt/spark-data/streaming/output"
)

results.show()

+--------+-------------------+-------+-----------+---------------+-----+-----+----------------+-------+
|event_id|          timestamp|user_id| event_type|      game_mode|level|score|duration_seconds|country|
+--------+-------------------+-------+-----------+---------------+-----+-----+----------------+-------+
|       6|2026-11-19 17:04:10| USR002|match_start|team_deathmatch|    3|    0|               0|     FR|
|       7|2026-11-19 17:05:30| USR003|match_start|  battle_royale|    2|    0|               0|     DE|
|      12|2026-11-19 17:10:00| USR004|match_start|  battle_royale|    2|    0|               0|     ES|
|      17|2026-11-19 17:15:10| USR005|match_start|  battle_royale|    3|    0|               0|     ES|
|      19|2026-11-19 17:17:00| USR006|match_start|team_deathmatch|    2|    0|               0|     FR|
|       3|2026-11-19 17:01:10| USR001|match_start|  battle_royale|    5|    0|               0|     ES|
+--------+-------------------+-------+-----------+--------------

In [22]:
import threading

stop_thread = threading.Thread(target=query.stop)
stop_thread.start()

In [ ]:
query.stop()

In [34]:
import os

input_path = "/opt/spark-data/streaming/incoming"

print(os.listdir(input_path))

['game_events_01.csv']


In [35]:
with open("/opt/spark-data/streaming/incoming/game_events_01.csv", "r") as f:
    print(f.read())

event_id,timestamp,user_id,event_type,game_mode,level,score,duration_seconds,country
1,2026-11-19 17:00:01,USR001,login,battle_royale,1,0,0,ES
2,2026-11-19 17:00:05,USR002,login,team_deathmatch,1,0,0,FR
3,2026-11-19 17:01:10,USR001,match_start,battle_royale,5,0,0,ES
4,2026-11-19 17:01:15,USR003,login,battle_royale,1,0,0,DE
5,2026-11-19 17:03:20,USR001,score,battle_royale,5,150,120,ES



In [45]:
import os

input_path = "/opt/spark-data/streaming/incoming"

for filename in os.listdir(input_path):
    filepath = os.path.join(input_path, filename)
    print(
        filename,
        "Size:", os.path.getsize(filepath),
        "bytes"
    )

game_events_01.csv Size: 387 bytes


In [46]:
for progress in query.recentProgress:
    print(
        "Batch:", progress["batchId"],
        "Input rows:", progress["numInputRows"]
    )

Batch: 0 Input rows: 0
Batch: 0 Input rows: 0
Batch: 0 Input rows: 0
Batch: 0 Input rows: 0


In [47]:
print("Active:", query.isActive)
print("Status:", query.status)
print("Exception:", query.exception())

Active: False
Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Exception: None


In [48]:
test_df = (
    spark.read
    .option("header", True)
    .schema(game_schema)
    .csv("/opt/spark-data/streaming/incoming")
)

test_df.show(truncate=False)
print("Rows:", test_df.count())

Py4JJavaError: An error occurred while calling o162.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Master removed our application: KILLED
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$3(DAGScheduler.scala:4568)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:4568)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:4560)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:4560)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1(DAGScheduler.scala:2054)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1$adapted(DAGScheduler.scala:2054)
	at scala.Option.foreach(Option.scala:437)
	at org.apache.spark.scheduler.DAGScheduler.handleTaskSetFailed(DAGScheduler.scala:2054)
	at org.apache.spark.s